# Pydantic v2 — Hands-on Examples

A practical notebook covering the main concepts from the Pydantic presentation:

- `BaseModel`
- Validation and `ValidationError`
- Type coercion and strict validation
- Optional fields and defaults
- Nested models and lists
- `Field` constraints
- Custom field/model validators
- Enums
- Serialization and aliases
- JSON Schema
- `model_validate()` / `model_validate_json()`
- `TypeAdapter`
- Generic models
- FastAPI-style request/response models
- LLM structured-output patterns

> **Recommended environment:** Python 3.10+ and Pydantic v2.


In [1]:
# Install dependencies if needed
%pip install -q "pydantic>=2,<3" email-validator


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pydantic
from pydantic import (
    BaseModel,
    Field,
    ValidationError,
    EmailStr,
    HttpUrl,
    ConfigDict,
    TypeAdapter,
    field_validator,
    model_validator,
)
from enum import Enum
from typing import Generic, TypeVar

print("Pydantic version:", pydantic.__version__)

## 1. Your first Pydantic model

`BaseModel` is the foundation of Pydantic. Type annotations become the model's data contract.

In [ ]:
class User(BaseModel):
    name: str
    age: int
    email: EmailStr


user = User(
    name="Ram",
    age=25,
    email="ram@example.com",
)

print(user)
print(user.name)
print(type(user.age))
print(type(user.email))

## 2. Validation errors

Pydantic validates data when the model is created. Invalid input raises `ValidationError` with structured error details.

In [ ]:
try:
    User(
        name="Ram",
        age="not-a-number",
        email="not-an-email",
    )
except ValidationError as exc:
    print(str(exc))
    print("\nStructured errors:")
    for error in exc.errors():
        print(error)

## 3. Type coercion

Pydantic can often convert reasonable input into the declared type.

For production systems where implicit conversion is undesirable, use strict validation.

In [ ]:
class Product(BaseModel):
    name: str
    price: float
    quantity: int


product = Product(
    name="Laptop",
    price="999.99",
    quantity="2",
)

print(product)
print(type(product.price))
print(type(product.quantity))

In [ ]:
class StrictProduct(BaseModel):
    model_config = ConfigDict(strict=True)

    name: str
    price: float
    quantity: int


try:
    StrictProduct(
        name="Laptop",
        price="999.99",
        quantity="2",
    )
except ValidationError as exc:
    print(exc)

## 4. Optional fields and defaults

A field such as `age: int | None = None` can be omitted and defaults to `None`.

A normal field such as `name: str` is required.

In [ ]:
class Profile(BaseModel):
    name: str
    age: int | None = None
    active: bool = True
    country: str = "India"


profile = Profile(name="Ram")

print(profile)
print(profile.model_dump())

## 5. Nested models

Pydantic recursively validates nested structures. This is especially useful for API payloads and complex JSON.

In [ ]:
class Address(BaseModel):
    city: str
    country: str


class Customer(BaseModel):
    name: str
    email: EmailStr
    address: Address


customer = Customer(
    name="Ram",
    email="ram@example.com",
    address={
        "city": "Mumbai",
        "country": "India",
    },
)

print(customer)
print(type(customer.address))
print(customer.address.city)

## 6. Lists of nested models

In [ ]:
class Item(BaseModel):
    name: str
    price: float
    quantity: int = 1


class Order(BaseModel):
    order_id: int
    customer: Customer
    items: list[Item]


order = Order(
    order_id=1001,
    customer={
        "name": "Ram",
        "email": "ram@example.com",
        "address": {
            "city": "Mumbai",
            "country": "India",
        },
    },
    items=[
        {"name": "Laptop", "price": 1000, "quantity": 1},
        {"name": "Mouse", "price": 50, "quantity": 2},
    ],
)

print(order)
print(order.items[0].name)

## 7. Field constraints

`Field()` lets you add validation constraints and metadata.

In [ ]:
class Registration(BaseModel):
    username: str = Field(min_length=3, max_length=30)
    age: int = Field(ge=18, le=100)
    password: str = Field(min_length=8)


valid = Registration(
    username="ram_ai",
    age=25,
    password="strongpass123",
)

print(valid)

try:
    Registration(
        username="x",
        age=12,
        password="short",
    )
except ValidationError as exc:
    for error in exc.errors():
        print(error)

## 8. Custom field validators

Use `field_validator()` when the built-in type constraints are not enough.

In [ ]:
class Account(BaseModel):
    username: str
    email: EmailStr

    @field_validator("username")
    @classmethod
    def username_must_not_contain_spaces(cls, value: str) -> str:
        if " " in value:
            raise ValueError("username cannot contain spaces")
        return value.lower()


print(Account(username="RamAI", email="ram@example.com"))

try:
    Account(username="Ram AI", email="ram@example.com")
except ValidationError as exc:
    print(exc)

## 9. Model-level validation

`model_validator()` is useful when a rule depends on multiple fields.

In [ ]:
class PasswordChange(BaseModel):
    password: str = Field(min_length=8)
    confirm_password: str = Field(min_length=8)

    @model_validator(mode="after")
    def passwords_must_match(self):
        if self.password != self.confirm_password:
            raise ValueError("passwords do not match")
        return self


print(
    PasswordChange(
        password="secret123",
        confirm_password="secret123",
    )
)

try:
    PasswordChange(
        password="secret123",
        confirm_password="different123",
    )
except ValidationError as exc:
    print(exc)

## 10. Enums for controlled values

Enums are useful when only a known set of values should be accepted.

In [ ]:
class Status(str, Enum):
    pending = "pending"
    approved = "approved"
    rejected = "rejected"


class Application(BaseModel):
    applicant: str
    status: Status


application = Application(
    applicant="Ram",
    status="approved",
)

print(application)
print(application.status)
print(application.status.value)

try:
    Application(applicant="Ram", status="banana")
except ValidationError as exc:
    print(exc)

## 11. Specialized types

Pydantic provides useful types such as `EmailStr` and `HttpUrl`.

In [ ]:
class Website(BaseModel):
    name: str
    url: HttpUrl
    owner_email: EmailStr


website = Website(
    name="Example",
    url="https://example.com",
    owner_email="owner@example.com",
)

print(website)
print(type(website.url))
print(type(website.owner_email))

## 12. Serialization

Pydantic models can be converted back into dictionaries and JSON.

In [ ]:
user = User(
    name="Ram",
    age=25,
    email="ram@example.com",
)

as_dict = user.model_dump()
as_json = user.model_dump_json()

print("Dictionary:")
print(as_dict)

print("\nJSON:")
print(as_json)

## 13. Aliases

Aliases are useful when your Python naming convention differs from an external API's field names.

In [ ]:
class APIUser(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    first_name: str = Field(alias="firstName")
    last_name: str = Field(alias="lastName")


api_user = APIUser.model_validate({
    "firstName": "Ram",
    "lastName": "Kumar",
})

print(api_user)
print("\nPython names:", api_user.model_dump())
print("\nAPI names:", api_user.model_dump(by_alias=True))

## 14. Extra fields: forbid unexpected input

For strict API contracts, you may want to reject fields that are not part of the schema.

In [ ]:
class StrictUser(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str
    age: int


try:
    StrictUser(
        name="Ram",
        age=25,
        unexpected="value",
    )
except ValidationError as exc:
    print(exc)

## 15. `model_validate()`

Use this when your input is already a Python dictionary or compatible Python object.

In [ ]:
data = {
    "name": "Ram",
    "age": 25,
    "email": "ram@example.com",
}

user = User.model_validate(data)

print(user)

## 16. `model_validate_json()`

Use this when the input is JSON text.

In [ ]:
json_data = '''
{
    "name": "Ram",
    "age": 25,
    "email": "ram@example.com"
}
'''

user = User.model_validate_json(json_data)

print(user)

## 17. JSON Schema

A Pydantic model can produce a machine-readable JSON Schema. This is a key bridge between Python types, APIs and structured AI workflows.

In [ ]:
schema = User.model_json_schema()

from pprint import pprint
pprint(schema)

## 18. `TypeAdapter`

`TypeAdapter` lets you validate arbitrary Pydantic-supported types without creating a `BaseModel`.

This is useful for things like `list[int]`, unions and other reusable type definitions.

In [ ]:
int_list_adapter = TypeAdapter(list[int])

print(int_list_adapter.validate_python([1, 2, 3]))

try:
    int_list_adapter.validate_python([1, "bad", 3])
except ValidationError as exc:
    print(exc)

## 19. Generic response models

Generics are useful when an API repeatedly wraps different data types in the same response structure.

In [ ]:
T = TypeVar("T")


class APIResponse(BaseModel, Generic[T]):
    success: bool
    data: T


user_response = APIResponse[User](
    success=True,
    data={
        "name": "Ram",
        "age": 25,
        "email": "ram@example.com",
    },
)

print(user_response)
print(type(user_response.data))
print(user_response.model_dump())

## 20. FastAPI-style request/response models

You can use the same Pydantic models as API contracts.

Typical architecture:

```text
HTTP JSON
   ↓
Pydantic request model
   ↓
Validated Python object
   ↓
Business logic
   ↓
Pydantic response model
   ↓
JSON response
```

The following is a FastAPI example. It is intentionally not executed here so the notebook remains focused on Pydantic itself.

In [ ]:
# Example FastAPI integration

fastapi_example = '''
from fastapi import FastAPI
from pydantic import BaseModel, EmailStr

app = FastAPI()

class CreateUser(BaseModel):
    name: str
    email: EmailStr
    age: int

class UserResponse(BaseModel):
    id: int
    name: str
    email: EmailStr

@app.post("/users", response_model=UserResponse)
def create_user(user: CreateUser):
    return UserResponse(
        id=1,
        name=user.name,
        email=user.email,
    )
'''

print(fastapi_example)

## 21. LLM structured-output pattern

LLMs produce probabilistic output. Your application usually needs deterministic structure.

A Pydantic model gives you a contract for the result:

```text
LLM output
    ↓
Parse
    ↓
Pydantic validation
    ↓
Valid structured object
    ↓
Application logic

Invalid
    ↓
Retry / repair / reject
```

This pattern is useful for extraction, classification, routing, tool arguments and other structured AI workflows.

In [ ]:
class SentimentResult(BaseModel):
    sentiment: str
    confidence: float = Field(ge=0, le=1)
    reason: str


# Imagine this dictionary came from an LLM structured-output call.
llm_output = {
    "sentiment": "positive",
    "confidence": 0.94,
    "reason": "The customer says the product is excellent.",
}

result = SentimentResult.model_validate(llm_output)

print(result)
print(result.model_dump())

## 22. A more constrained LLM schema

For classification, an enum is often better than a free-form string.

In [ ]:
class Sentiment(str, Enum):
    positive = "positive"
    neutral = "neutral"
    negative = "negative"


class SentimentResultV2(BaseModel):
    sentiment: Sentiment
    confidence: float = Field(ge=0, le=1)
    reason: str


result = SentimentResultV2.model_validate({
    "sentiment": "positive",
    "confidence": 0.94,
    "reason": "The customer is happy.",
})

print(result)
print(result.sentiment.value)

## 23. Mini-project: validate an e-commerce order

Let's combine nested models, enums, constraints and custom validation.

In [ ]:
class OrderStatus(str, Enum):
    pending = "pending"
    paid = "paid"
    shipped = "shipped"
    cancelled = "cancelled"


class OrderItem(BaseModel):
    product_id: int = Field(gt=0)
    name: str = Field(min_length=1)
    unit_price: float = Field(gt=0)
    quantity: int = Field(gt=0, le=100)


class ShippingAddress(BaseModel):
    line1: str = Field(min_length=1)
    city: str = Field(min_length=2)
    country: str = Field(min_length=2)


class EcommerceOrder(BaseModel):
    order_id: int = Field(gt=0)
    customer_email: EmailStr
    status: OrderStatus
    items: list[OrderItem] = Field(min_length=1)
    shipping_address: ShippingAddress

    @model_validator(mode="after")
    def validate_order(self):
        if self.status == OrderStatus.shipped and not self.items:
            raise ValueError("A shipped order must contain at least one item")
        return self

    @property
    def total(self) -> float:
        return sum(item.unit_price * item.quantity for item in self.items)


order = EcommerceOrder(
    order_id=5001,
    customer_email="ram@example.com",
    status="paid",
    items=[
        {
            "product_id": 101,
            "name": "Laptop",
            "unit_price": 1000,
            "quantity": 1,
        },
        {
            "product_id": 102,
            "name": "Mouse",
            "unit_price": 50,
            "quantity": 2,
        },
    ],
    shipping_address={
        "line1": "123 Example Street",
        "city": "Mumbai",
        "country": "India",
    },
)

print(order)
print("Order total:", order.total)
print("Serialized:", order.model_dump())

## 24. Debugging validation failures

When working with APIs or LLMs, inspect `errors()` instead of only printing the exception.

In [ ]:
bad_order = {
    "order_id": -1,
    "customer_email": "bad-email",
    "status": "unknown",
    "items": [
        {
            "product_id": 0,
            "name": "",
            "unit_price": -10,
            "quantity": 0,
        }
    ],
    "shipping_address": {
        "line1": "",
        "city": "M",
        "country": "",
    },
}

try:
    EcommerceOrder.model_validate(bad_order)
except ValidationError as exc:
    for error in exc.errors():
        print(
            "Location:", error["loc"],
            "| Type:", error["type"],
            "| Message:", error["msg"],
        )

# Final mental model

Pydantic is easiest to remember as a **runtime data-contract layer**:

```text
Untrusted / external data
        ↓
   Pydantic model
        ↓
 Parse + validate
        ↓
 Typed Python object
        ↓
 Application logic
        ↓
 model_dump() / JSON
        ↓
 External system
```

For AI engineering, add:

```text
LLM
 ↓
Structured output
 ↓
Pydantic validation
 ↓
Typed application data
 ↓
Business logic
```

### Suggested next exercises

1. Add a `discount` field and calculate the final order total.
2. Add currency validation with an enum.
3. Add a validator that rejects disposable email domains.
4. Generate JSON Schema for the order model.
5. Build the same models into a small FastAPI service.
6. Use a real LLM structured-output API and validate the result with Pydantic.
